Exercise: Build a document Q&A system



In [ ]:
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()


def exercise_document_qa():
    """
    EXERCISE: Build a complete document Q&A system that:
    1. Takes a text document as input
    2. Splits and embeds it
    3. Allows multiple questions
    4. Returns answers with confidence scores
    """

    class DocumentQA:
        def __init__(self, document: str, source_name: str = "document"):
            # Split document
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=500,
                chunk_overlap=50
            )

            doc = Document(
                page_content=document,
                metadata={"source": source_name}
            )

            chunks = splitter.split_documents([doc])

            # Create vector store
            self.vectorstore = Chroma.from_documents(
                documents=chunks,
                embedding=OpenAIEmbeddings(
                    model="text-embedding-3-small"
                ),
            )

            # Create retriever
            self.retriever = self.vectorstore.as_retriever(
                search_kwargs={"k": 3}
            )

            # Create LLM
            self.llm = init_chat_model(
                model="gpt-4o-mini",
                temperature=0.2
            )

            # Create prompt
            self.prompt = ChatPromptTemplate.from_template(
                """
                Answer the question based only on the provided context.

                Context:
                {context}

                Question:
                {question}

                Format your response as:

                [Confidence: High/Medium/Low]
                Answer: <your answer>
                """
            )

            # Helper function
            def format_docs(docs):
                return "\n\n".join(doc.page_content for doc in docs)

            # Build chain 
            #In future when invoked then LangChain automatically sends the same input to both branches simultaneously(conrext branch and question branch)
            self.chain = (
                {
                    "context": self.retriever | format_docs, #It searches the vector database and returns a list of Document object. Those documents immediately become the input to format_docs(), So the context branch produces one big string.
                    "question": RunnablePassthrough(),
                }
                | self.prompt #The context and question are substituted into the prompt template, which becomes the message sent to the LLM.
                | self.llm
                | StrOutputParser()
            )

        def ask(self, question: str) -> str:
            return self.chain.invoke(question)

    # Sample document
    test_doc = """
    The Python programming language was created by Guido van Rossum.
    First released in 1991, Python emphasizes code readability.
    Python 3.12 was released in October 2023 with improved error messages.
    The language is named after Monty Python, not the snake.
    """

    qa = DocumentQA(test_doc, "python_facts")

    print("Document Q&A System:\n")

    questions = [
        "Who created Python?",
        "When was Python 3.12 released?",
        "Why is Python named Python?",
    ]

    for q in questions:
        answer = qa.ask(q)
        print(f"Q: {q}")
        print(f"A: {answer}\n")


if __name__ == "__main__":
    exercise_document_qa()